In [2]:
import xgboost as xgb
import model_XGBoost as xgb_model

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dateutil.relativedelta import relativedelta
import datetime as dt
import holidays
holidays_de= holidays.Germany()

from sklearn.metrics import root_mean_squared_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
import scipy.stats as stats

plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100

ImportError: cannot import name 'root_mean_squared_error' from 'sklearn.metrics' (C:\Users\bened\Documents\neuefische\capstone-project\ai-for-energy\.venv\Lib\site-packages\sklearn\metrics\__init__.py)

In [ ]:
df_all = pd.read_csv('../../data_cleaned/merged/02_4_clean_data.csv')
df_all.head()

In [ ]:
df_years = df_all.query('2019 <= year').copy()
df_years.head()

In [ ]:
df_years = pd.get_dummies(df_years, columns=['year'], drop_first=True)
df_years.head()

In [ ]:
df_years.drop(columns=['period_end_utc'], inplace=True)

In [ ]:
df_years["period_start_utc"] = pd.to_datetime(df_years["period_start_utc"], errors="coerce")
df_years["period_start_utc"] = df_years["period_start_utc"].apply(lambda x: x.replace(tzinfo=None))
df_years["date"] = pd.to_datetime(df_years["date"], errors="coerce")

In [ ]:
df_years.info()

In [ ]:
df_years.head()

In [ ]:
df_years.set_index('period_start_utc', inplace=True)

In [ ]:
df_years.info()

In [ ]:
df_years.plot( y='price', style='.', figsize=(15, 5), color='blue', title='Price')
plt.show()

# Train / Test Split

In [ ]:
start_date = pd.to_datetime('2019-01-01 00:00:00')
split_date = pd.to_datetime('2025-01-01 00:00:00')
end_date = pd.to_datetime('2026-01-01 00:00:00')

In [3]:
train = df_years.query('@start_date <= date < @split_date')
test = df_years.query('@split_date <= date < @end_date')

#train = df_years.query('date < 2021')
#test = df_years.query('2021 <= date < "2021-07"')

fig, ax = plt.subplots(figsize=(15, 5))
train.plot(y='price', ax=ax, label='Training Set', title='Training/Test Split Set')
test.plot(y='price', ax=ax, label = 'Testing Set')
ax.axvline(split_date, color='red', linestyle='dashed', linewidth=2)
plt.legend()
plt.show()

NameError: name 'df_years' is not defined

# Feature Creation

In [ ]:
df_years.index

In [ ]:
def create_features(df):
    """
    create time series features based on time series index
    :param df:
    :return:
    """
    df = df.copy()

    df['hour'] = df.index.hour
    df['dayofweek'] = df.index.dayofweek
    df['weekday'] = df.index.weekday
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    return df

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
sns.boxplot(data = df_years, x = 'hour', y= 'price')
ax.set_title('Price by Hour')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
sns.boxplot(data = df_years, x = 'month', y= 'price', palette='Blues')
ax.set_title('Price by Month')

# Create Model

In [ ]:
df_years.columns

In [ ]:
#FEATURES = ['year', 'month', 'day', 'dayofyear', 'hour', 'week', 'dayofweek', 'load_forecast_da', 'res_sum_da', 'dayofyear_sin1',
#       'dayofyear_cos1', 'hour_sin1', 'hour_cos1', 'dayofweek_sin1',
#       'dayofweek_cos1', 'is_holiday', 'price_gas']
FEATURES = ['load_forecast_da', 'res_sum_da', 'gen_forecast_da', 'dayofyear_sin1',
       'dayofyear_cos1', 'hour_sin1', 'hour_cos1', 'dayofweek_sin1',
       'dayofweek_cos1', 'is_holiday', 'price_gas', 'year_2020', 'year_2021', 'year_2022',
       'year_2023', 'year_2024', 'year_2025']
#FEATURES = ['year', 'price_gas', 'res_sum_da', 'load_forecast_da']
TARGET = 'price'

In [ ]:
reg = xgb_model.create_and_train_XGBoost(train, test, FEATURES, TARGET)

In [ ]:
#reg.score()

# Feature Importance

In [ ]:
xgb_model.display_feature_importance(reg)

# Forecast on Test

In [ ]:
test['prediction'] = reg.predict(test[FEATURES])

In [ ]:
df_years = df_years.merge(test[['prediction']],  how = 'left', left_index=True, right_index=True) # merge on index columns

In [ ]:
df_years

In [ ]:
df_years.info()

In [ ]:
ax = df_years[['price']].plot(figsize=(15, 5))
df_years['prediction'].plot(ax=ax, style = '.')
plt.legend(['test data', 'prediction'])
plt.show()
ax.set_title('Test Data and Predictions')

# smaller test frames

In [ ]:
add_three_months = relativedelta(months=3)
add_one_week = relativedelta(days=7)

In [ ]:
first_day_four_months_in = split_date + add_three_months
first_day_four_months_in.to_datetime64()
last_day_four_months_in = first_day_four_months_in + add_one_week
last_day_four_months_in.to_datetime64()

ax = df_years.loc[(first_day_four_months_in < df_years.index) & (df_years.index < last_day_four_months_in)]['price'].plot(figsize=(15, 5), title='Predictions - 7 days of data')
df_years.loc[(first_day_four_months_in < df_years.index) & (df_years.index < last_day_four_months_in)]['prediction'].plot(ax=ax, style='.')
plt.legend(['test data', 'prediction'])
plt.show()

In [ ]:
ax = df_years.loc[("2020-04-01" < df_years.index) & (df_years.index < "2020-04-10")]['price'].plot(figsize=(15, 5), title='Predictions - 10 days of data')
plt.legend(['test data'])
plt.show()

In [ ]:
split_date_one_week_in = split_date + add_one_week

ax = df_years.loc[(split_date < df_years.index) & (df_years.index < split_date_one_week_in)]['price'].plot(figsize=(15, 5), title='Predictions - 7 days of data')
df_years.loc[(split_date < df_years.index) & (df_years.index < split_date_one_week_in)]['prediction'].plot(ax=ax, style='.')
plt.legend(['test data', 'prediction'])
plt.show()

# Using Grid/Randomizer Search

In [ ]:
# Define the hyperparameter distributions
param_grid = {
    'base_score':[0.5],
    'max_depth': np.arange(3, 8, 1).tolist(),
    'learning_rate': np.arange(0.02, 0.2, 0.04).tolist(),
    'subsample': [0.5, 0.7, 1],
    'n_estimators': np.arange(500, 2000, 500).tolist(),
    'colsample_bytree': np.arange(0.7, 0.95, 0.1).tolist()
}

In [ ]:
# Define the hyperparameter distributions
param_grid = {
    'base_score':[0.5],
    'max_depth': np.arange(5, 7, 1).tolist(),
    'learning_rate': np.arange(0.01, 0.2, 0.1).tolist(),
    'subsample': [0.8],
    'n_estimators': np.arange(1000, 2000, 500).tolist(),
    'colsample_bytree': [0.8], #np.arange(0.7, 0.9, 0.1).tolist()
    'objective':['reg:squarederror'],
}

In [4]:
best_grid_search_model = xgb_model.paramater_search_XGBoost(train, FEATURES, TARGET, 'grid', param_grid)

NameError: name 'train' is not defined

In [ ]:
y_pred = best_grid_search_model(test[FEATURES])
rmse_grid_search = root_mean_squared_error(test[TARGET], y_pred)

In [ ]:
# Define the hyperparameter distributions
param_dist = {
    'base_score':stats.uniform(0.3, 0.7),
    'max_depth': stats.randint(3, 10),
    'learning_rate': stats.uniform(0.001, 0.2),
    'subsample': stats.uniform(0.5, 0.9),
    'n_estimators':stats.randint(200, 2000),
    'colsample_bytree':stats.uniform(0.5, 0.8)
}

In [ ]:
#xgb_model.paramater_search_XGBoost(train, FEATURES, TARGET, 'random', param_dist)